In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import torch

CODE = next(
    path.parent.parent
    for path in Path("/kaggle/input").rglob("day6.py")
    if path.parent.name == "octlm"
)
OUT = Path("/kaggle/working/day6")
OUT.mkdir(parents=True, exist_ok=True)
MODEL = OUT / "qwen"
if not torch.cuda.is_available() or torch.cuda.get_device_capability(0) != (7, 5):
    raise RuntimeError("Select a Kaggle T4 GPU")
print(
    "Code:", CODE, "PyTorch:", torch.__version__, "GPU:", torch.cuda.get_device_name(0), flush=True
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "transformers==4.57.6",
        "tokenizers==0.22.2",
        "jinja2>=3.1.6,<4",
    ],
    check=True,
)

In [ ]:
def run(stage, *arguments):
    subprocess.run(
        [sys.executable, "-m", "octlm.day6", stage, "--model", str(MODEL), *arguments],
        cwd=CODE,
        check=True,
    )


subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests"],
    cwd=CODE,
    check=True,
    env={**os.environ, "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2"},
)
run("prepare")
run("dry-run", "--out", str(OUT / "dry-run.jsonl"))

In [ ]:
run(
    "run",
    "--device",
    "cuda:0",
    "--tokens",
    "32",
    "--repeats",
    "3",
    "--out",
    str(OUT / "results.jsonl"),
)
print((OUT / "results.jsonl").read_text(), flush=True)